# Responses to the stimulation trains in one session

The figures of Hadjas et al. (2025), [*Parkinson's disease-vulnerable and -resilient dopamine neurons display opposite responses to excitatory input*](https://www.biorxiv.org/content/10.1101/2025.06.03.657460v1) show dF/F aligned on the onset of the stimulation trains and averaged by train duration and pulse frequency. This notebook makes these plots for one session of [DANDI:001528](https://dandiarchive.org/dandiset/001528/draft).

The default session records SNr GABA neurons while PPN inputs are stimulated at 5, 10, 20 and 40 Hz, with trains of 250 ms, 1 s and 4 s. Any session of the dandiset can be used, for example `sub-C3762/sub-C3762_ses-varying-frequencies.nwb` for dopamine release in the dorsolateral striatum during STN stimulation.

In [ ]:
%matplotlib inline
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import remfile
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

In [ ]:
DANDISET_ID = "001528"
DANDISET_VERSION = "draft"


def open_session(asset_path):
    """Open an NWB file of the dandiset, streamed from the DANDI Archive."""
    with DandiAPIClient() as client:
        asset = client.get_dandiset(DANDISET_ID, DANDISET_VERSION).get_asset_by_path(asset_path)
        url = asset.get_content_url(follow_redirects=1, strip_query=True)
    file = remfile.File(url, disk_cache=remfile.DiskCache("remfile_cache"))
    return NWBHDF5IO(file=h5py.File(file, "r"), load_namespaces=True)

## Read dF/F and the trains

A varying-frequencies session is recorded in one TDT block per train duration. The dF/F series holds the blocks one after the other, with their timestamps, so its times are on the clock of the trains.

In [ ]:
asset_path = "sub-C4561/sub-C4561_ses-varying-frequencies.nwb"

io = open_session(asset_path)
nwbfile = io.read()
dff_name = next(name for name in nwbfile.processing["ophys"].data_interfaces if name.startswith("deltaF_over_F"))
dff = nwbfile.processing["ophys"][dff_name]
dff_values, dff_times = dff.data[:], dff.get_timestamps()
trains = nwbfile.intervals["optogenetic_epochs_table"].to_dataframe()
print(f"{nwbfile.subject.subject_id}, {dff_name}: {len(trains)} trains")
trains.groupby(["train_duration", "stimulus_frequency"]).size().unstack()

## Align dF/F on the trains

Each trial is the dF/F from 5 s before to 10 s after the onset of a train, minus its mean over the second before the onset, as in the lab's MATLAB code. The trials below are the same as those of the lab's code; `hadjas_2025.analysis.trial_analysis` in [hnasko-lab-to-nwb](https://github.com/catalystneuro/hnasko-lab-to-nwb) also computes the statistics of the lab's time windows.

In [ ]:
RATE = 100.0  # Hz, rate of dF/F within a block
trial_offsets = np.arange(-500, 1001)  # samples from the onset sample, -5 s to 10 s
trial_time = trial_offsets / RATE
baseline = (trial_offsets >= -101) & (trial_offsets < 0)  # the 101 samples before the onset sample

# The lab's code takes as onset sample floor(100 * onset time from the block start), counted from 1,
# which is two samples before the first sample after the onset
onset_indices = np.searchsorted(dff_times, trains["start_time"].to_numpy()) - 2
trials = dff_values[onset_indices[:, np.newaxis] + trial_offsets]
trials = trials - trials[:, baseline].mean(axis=1, keepdims=True)
trials.shape

## Every trial

One row per train, grouped by train duration and pulse frequency. The dashed line marks the onset.

In [ ]:
durations = ["250ms", "1s", "4s"]
duration_in_seconds = {"250ms": 0.25, "1s": 1.0, "4s": 4.0}
frequencies = sorted(trains["stimulus_frequency"].unique())
# Sort the trials by train duration, then pulse frequency
order = np.lexsort((trains["stimulus_frequency"], trains["train_duration"].map(durations.index)))
conditions = (
    trains.iloc[order].reset_index(drop=True).groupby(["train_duration", "stimulus_frequency"], sort=False).indices
)
limit = np.percentile(np.abs(trials), 99)

figure, ax = plt.subplots(figsize=(8, 7), layout="constrained")
image = ax.imshow(
    trials[order],
    aspect="auto",
    cmap="RdBu_r",
    vmin=-limit,
    vmax=limit,
    extent=[trial_time[0], trial_time[-1], len(order), 0],
    interpolation="none",
)
ax.axvline(0, color="0.2", ls="--", lw=1)
for rows in conditions.values():
    ax.axhline(rows[-1] + 1, color="0.5", lw=0.5)
ax.set_yticks(
    [rows.mean() + 0.5 for rows in conditions.values()], [f"{d}, {f:g} Hz" for d, f in conditions], fontsize=7
)
ax.set(xlabel="time from train onset (s)", title=f"{nwbfile.subject.subject_id}, {dff_name}")
figure.colorbar(image, ax=ax, shrink=0.6, label="dF/F minus baseline (%)");

## Mean responses

Mean and standard error over the 5 trains of each condition, one panel per train duration, darker lines for higher frequencies. The gray band is the train.

In [ ]:
colors = plt.cm.viridis_r(np.linspace(0.15, 0.9, len(frequencies)))
figure, axes = plt.subplots(1, len(durations), figsize=(13, 3.5), sharey=True, layout="constrained")
for ax, duration in zip(axes, durations):
    ax.axvspan(0, duration_in_seconds[duration], color="0.9", lw=0)
    for color, frequency in zip(colors, frequencies):
        condition = trials[
            ((trains["train_duration"] == duration) & (trains["stimulus_frequency"] == frequency)).to_numpy()
        ]
        mean = condition.mean(axis=0)
        sem = condition.std(axis=0, ddof=1) / np.sqrt(len(condition))
        ax.fill_between(trial_time, mean - sem, mean + sem, color=color, alpha=0.2, lw=0)
        ax.plot(trial_time, mean, color=color, lw=1.5, label=f"{frequency:g} Hz")
    ax.set(title=f"{duration} trains", xlabel="time from train onset (s)", xlim=(-2, 8))
axes[0].set_ylabel("dF/F minus baseline (%)")
axes[-1].legend(fontsize=8, loc="upper right");

## Mean dF/F during the train

The mean of each trial over the train, as in the summary panels of the paper.

In [ ]:
during_train = pd.DataFrame(
    {
        "train_duration": trains["train_duration"].to_numpy(),
        "stimulus_frequency": trains["stimulus_frequency"].to_numpy(),
        "mean_dff": [
            trial[(trial_time > 0) & (trial_time <= duration_in_seconds[duration])].mean()
            for trial, duration in zip(trials, trains["train_duration"])
        ],
    }
)
figure, axes = plt.subplots(1, len(durations), figsize=(10, 3), sharey=True, layout="constrained")
for ax, duration in zip(axes, durations):
    condition = during_train[during_train["train_duration"] == duration]
    positions = condition["stimulus_frequency"].map(frequencies.index)
    ax.plot(positions, condition["mean_dff"], "o", color="0.6", ms=4)
    means = condition.groupby("stimulus_frequency")["mean_dff"].mean()
    ax.plot(range(len(means)), means.values, "-o", color="0.1", lw=1.5)
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set_xticks(range(len(frequencies)), [f"{f:g}" for f in frequencies])
    ax.set(title=f"{duration} trains", xlabel="pulse frequency (Hz)")
axes[0].set_ylabel("mean dF/F during train (%)");

In [ ]:
io.close()